# Étape 3 — Nettoyage, déduplication et découpage
**Projet : Les LLM parlent-ils maghrébin ?** · Rôle Données

Entrée : `data/interim/unified_v0.parquet` (produit par `00_donnees.ipynb`).
Sorties dans `data/processed/`, **un Parquet par usage** :

| fichier | contenu | sert à |
|---|---|---|
| `sentiment.parquet` | DZ / TN, labels `neg`/`neu`/`pos`, splits train/val/test | Exp. 4 |
| `fakenews.parquet` | FASSILA fake news | bonus Exp. 4 |
| `lm.parquet` | textes pour le pré-entraînement (MA surtout), splits train/val/test | Exp. 3 |
| `parallel_eval.parquet` | FLORES+ (MA, TN, MSA, FR, EN), **évaluation uniquement** | Exp. 1 |
| `tokenizer_balanced.parquet` | corpus **équilibré par dialecte** (même nombre de mots), train uniquement | Exp. 2 |

Ce que fait le notebook, dans l'ordre :
1. corrige **DarijaBridge** (99 % de doublons) et retrouve la **source** des textes d'**Atlaset** ;
2. **normalise** : URL, pseudos, e-mails, numéros, emojis, lettres répétées, diacritiques et variantes de l'alef ;
3. sépare **arabizi / français / anglais** parmi les textes en lettres latines ;
4. retire les données **synthétiques** d'Atlaset du pré-entraînement ;
5. **déduplique** (et supprime les textes avec deux labels contradictoires) ;
6. **découpe** en train / val / test (seed 42), sans fuite entre les splits ;
7. construit le corpus **équilibré** pour le tokenizer.

**À la fin, copie les tableaux récapitulatifs et colle-les à Claude.**

## 0. Connexion et chargement

In [ ]:
!pip -q install datasets pyarrow scikit-learn
from google.colab import drive
drive.mount('/content/drive')

In [ ]:
# Hugging Face : nécessaire pour relire Atlaset (dataset protégé). Suis le lien et entre le code affiché.
from huggingface_hub import login
login()

In [ ]:
import re, gc, unicodedata
from pathlib import Path
import numpy as np
import pandas as pd
from datasets import load_dataset
from sklearn.model_selection import train_test_split

SEED = 42
ROOT = Path('/content/drive/MyDrive/darija_project')
INTERIM = ROOT / 'data' / 'interim'
PROC = ROOT / 'data' / 'processed'
PROC.mkdir(parents=True, exist_ok=True)

COLS = ['id', 'text', 'dialect', 'script', 'task', 'label', 'label_orig',
        'dataset', 'sub_source', 'orig_split', 'group_id', 'usage']
U = pd.read_parquet(INTERIM / 'unified_v0.parquet', columns=COLS)
print(U.shape)
print(U.groupby('dataset').size())

## 1. Outils
- `ar_ratio` : part de lettres arabes dans un texte (sert à choisir la bonne colonne).
- `detect_script` : même règle qu'à l'étape 2 (`arab` / `latin` / `mixed` / `other`).

In [ ]:
AR = re.compile('[؀-ۿݐ-ݿࢠ-ࣿﭐ-﷿ﹰ-﻿]')
LAT = re.compile('[A-Za-zÀ-ÿ]')

def ar_ratio(series, n=500):
    s = ' '.join(series.dropna().astype(str).head(n))
    letters = len(AR.findall(s)) + len(LAT.findall(s))
    return len(AR.findall(s)) / max(letters, 1)

def detect_script(text):
    a, l = len(AR.findall(text)), len(LAT.findall(text))
    if a + l == 0:
        return 'other'
    r = a / (a + l)
    return 'arab' if r >= 0.8 else 'latin' if r <= 0.2 else 'mixed'

def describe_columns(df):
    rows = []
    for c in df.columns:
        if df[c].dtype == object:
            s = df[c].dropna().astype(str)
            rows.append({'colonne': c, 'uniques': s.nunique(), 'part_arabe': round(ar_ratio(s), 2),
                         'exemple': s.iloc[0][:60] if len(s) else ''})
    return pd.DataFrame(rows).sort_values('uniques', ascending=False)

## 2. DarijaBridge : vérifier la colonne
À l'étape 2, 1 227 504 lignes sur 1 235 091 étaient des doublons. DarijaBridge est un corpus de **traduction** :
la même phrase peut revenir plusieurs fois, ou la colonne choisie automatiquement n'était pas la bonne.
On affiche toutes les colonnes : **la darija est la colonne en arabe avec le plus de valeurs uniques**.

In [ ]:
dbr = load_dataset('M-A-D/DarijaBridge', split='train').to_pandas()
print(dbr.shape)
display(describe_columns(dbr))

In [ ]:
cand = describe_columns(dbr)
cand = cand[cand.part_arabe >= 0.5]
DB_COL = cand.iloc[0]['colonne']          # ← corrige à la main si l'affichage ci-dessus montre une autre colonne
print('Colonne darija retenue :', DB_COL)

db_new = pd.DataFrame({'text': dbr[DB_COL].astype(str), 'dialect': 'MA', 'task': 'lm',
                       'dataset': 'darijabridge', 'orig_split': 'all', 'usage': 'train_ok'})
print('Textes uniques :', db_new.text.nunique(), 'sur', len(db_new))
del dbr; gc.collect()

## 3. Atlaset : retrouver la source de chaque texte
À l'étape 2, la colonne `source` n'a pas été trouvée. On cherche une colonne texte avec **peu de valeurs différentes**
(le nom du jeu de données d'origine) et on l'affiche pour vérification.

In [ ]:
at = load_dataset('atlasia/Atlaset')
at_df = pd.concat([at[s].to_pandas().assign(orig_split=s) for s in at.keys()], ignore_index=True)
del at; gc.collect()
print(at_df.shape, at_df.columns.tolist())
display(describe_columns(at_df))

In [ ]:
AT_TEXT = 'text' if 'text' in at_df.columns else describe_columns(at_df).iloc[0]['colonne']
desc = describe_columns(at_df.drop(columns=[AT_TEXT, 'orig_split']))
low = desc[(desc.uniques >= 2) & (desc.uniques <= 2000)]
AT_SRC = low.iloc[0]['colonne'] if len(low) else None   # ← corrige à la main si besoin
print('texte :', AT_TEXT, '| source :', AT_SRC)

at_new = pd.DataFrame({'text': at_df[AT_TEXT].astype(str), 'dialect': 'MA', 'task': 'lm', 'dataset': 'atlaset',
                       'orig_split': at_df['orig_split'].replace({'validation': 'dev'}), 'usage': 'train_ok'})
at_new['sub_source'] = at_df[AT_SRC].astype(str) if AT_SRC else 'inconnue'
del at_df; gc.collect()
pd.set_option('display.max_rows', 200)
display(at_new.sub_source.value_counts().to_frame('lignes'))

### Sources synthétiques ou traduites automatiquement
Les sources dont le nom contient un des mots-clés ci-dessous sont **exclues du pré-entraînement** (colonne `keep_lm = False`).
**Lis la liste affichée** : si une source authentique est exclue à tort, ou une source synthétique oubliée, modifie `SYNTH_KEYS`.

In [ ]:
SYNTH_KEYS = ['synth', 'gemini', 'gpt', 'sharegpt', 'flan', 'alpaca', 'dolly', 'oasst', 'instruct',
              'translat', 'ultrachat', 'tulu', 'generated', 'llm']
src_lower = at_new.sub_source.str.lower()
at_new['synthetic'] = src_lower.apply(lambda s: any(k in s for k in SYNTH_KEYS))
print(at_new.groupby(['sub_source', 'synthetic']).size().sort_values(ascending=False).to_string())
print('\nLignes exclues :', int(at_new.synthetic.sum()), 'sur', len(at_new))

In [ ]:
# On remplace les anciennes lignes DarijaBridge et Atlaset par les versions corrigées
U = U[~U.dataset.isin(['darijabridge', 'atlaset'])]
U = pd.concat([U, db_new, at_new], ignore_index=True)
U['synthetic'] = U['synthetic'].fillna(False).astype(bool)
del db_new, at_new; gc.collect()
print(U.groupby('dataset').size())

## 4. Normalisation
Choix (à justifier dans le rapport) :
- **NFKC** : ramène les formes de présentation arabes (ﻻ, ﺑ…) aux lettres normales ;
- **suppression** : URL, e-mails, `@pseudos`, numéros de téléphone, balises HTML, `#` des hashtags (le mot est gardé) ;
- **emojis** : supprimés, **sauf pour le sentiment** où ils portent de l'information ;
- **lettres répétées** : 3 fois ou plus → 2 (« bravooooo » → « bravoo », « ههههه » → « هه ») ;
- **arabe** : suppression des diacritiques et du tatweel (ـ), alef أ إ آ ٱ → ا. On garde ة et ى (choix prudent) ;
- **latin** : mis en minuscules ; les chiffres de l'arabizi (3, 7, 9…) sont conservés.

In [ ]:
RULES = [
    (r'<[^>]+>|&[a-z]+;', ' '),                         # HTML
    (r'https?://\S+|www\.\S+', ' '),                   # URL
    (r'\S+@\S+\.\S+', ' '),                            # e-mails
    (r'@\w+', ' '),                                    # pseudos
    (r'(?<!\w)\+?\d[\d .-]{7,}\d(?!\w)', ' '),         # numéros (8 chiffres ou plus)
    (r'#', ' '),                                       # hashtags : on garde le mot
    ('[ً-ٰٟۖ-ۭـ]', ''),  # diacritiques + tatweel
    ('[آأإٱ]', 'ا'),          # variantes de l'alef
    (r'(.)\1{2,}', r'\1\1'),                           # lettres répétées
]
EMOJI = '[\U0001F000-\U0001FAFF☀-➿⬀-⯿️‍⃣]'

def normalize(s: pd.Series, keep_emoji: bool) -> pd.Series:
    s = s.astype(str).map(lambda t: unicodedata.normalize('NFKC', t)).str.lower()
    for pat, rep in RULES:
        s = s.str.replace(pat, rep, regex=True)
    if not keep_emoji:
        s = s.str.replace(EMOJI, ' ', regex=True)
    return s.str.replace(r'\s+', ' ', regex=True).str.strip()

is_sent = U.task.eq('sentiment')
U.loc[is_sent, 'text'] = normalize(U.loc[is_sent, 'text'], keep_emoji=True)
U.loc[~is_sent, 'text'] = normalize(U.loc[~is_sent, 'text'], keep_emoji=False)
U['script'] = U['text'].map(detect_script)
print(U.groupby(['dataset', 'script']).size().unstack(fill_value=0))

## 5. Arabizi, français ou anglais ?
Parmi les textes en lettres latines (TUNIZI, TSAC, une partie de l'algérien), on devine la langue avec des règles simples :
- **arabizi** : chiffres collés à des lettres (« 3lach », « 7aja », « 9al ») ou mots darija fréquents ;
- **français / anglais** : au moins 30 % de mots outils (« le, de, est… » / « the, is, and… »).
Résultat dans la colonne `lang` (`ar`, `arabizi`, `fr`, `en`, `mixed`, `latin_autre`). Les textes arabes ont `lang = ar`.

In [ ]:
FR = set('le la les de des du un une et est que qui pas pour dans ce cette il elle nous vous je tu sur avec mais très plus au aux ou on son sa ses mon ma mes été être avoir fait cest jai'.split())
EN = set('the and is are you to of in it this that for with not was be have i my your so but what'.split())
ARZ_WORDS = set('ya w fi el ma la bech barcha mouch ena enti inti kifech chnowa wallah walah inchallah rabi yarabi allah 3la ala lel bel men min hadi hada rani raho wach kima bezaf bzaf chwiya ychouf nchallah mabrouk saha khoya'.split())
ARZ_DIGIT = re.compile(r'[a-z][2356789]|[2356789][a-z]')

def guess_latin(t):
    toks = re.findall(r"[a-z0-9à-ÿ']+", t)
    if not toks:
        return 'latin_autre'
    n = len(toks)
    arz = ARZ_DIGIT.search(t) is not None or sum(w in ARZ_WORDS for w in toks) / n >= 0.2
    fr = sum(w in FR for w in toks) / n
    en = sum(w in EN for w in toks) / n
    if fr >= 0.3 and fr > en:
        return 'fr'
    if en >= 0.3:
        return 'en'
    return 'arabizi' if arz else 'latin_autre'

U['lang'] = np.select([U.script.eq('arab'), U.script.eq('mixed')], ['ar', 'mixed'], default='latin_autre')
lat = U.script.eq('latin') & ~U.dataset.eq('flores_plus')
U.loc[lat, 'lang'] = U.loc[lat, 'text'].map(guess_latin)
U.loc[U.dataset.eq('flores_plus'), 'lang'] = U.loc[U.dataset.eq('flores_plus'), 'dialect'].str.lower()
print(pd.crosstab(U.dataset, U.lang))
for l in ['arabizi', 'fr', 'en']:
    print(f'\n--- exemples {l} ---')
    print(U[U.lang.eq(l)].text.sample(min(5, int(U.lang.eq(l).sum())), random_state=SEED).to_string())

## 6. Filtres et déduplication
- textes vides ou sans aucune lettre : supprimés ;
- pré-entraînement : au moins 3 mots, et pas de données synthétiques ;
- sentiment / fake news : un même texte avec **deux labels différents** dans un dataset → supprimé (on ne sait pas lequel est juste) ;
- doublons exacts : supprimés (par dataset et tâche pour le sentiment, sur tout le corpus pour le pré-entraînement) ;
- **contamination** : tout texte identique à une phrase de FLORES+ est retiré des données d'entraînement.

In [ ]:
n0 = len(U)
U = U[U.text.str.contains('[^\\W\\d_]', regex=True)]
U['n_words'] = U.text.str.count(r'\S+')
lm = U.task.eq('lm')
U = U[~(lm & ((U.n_words < 3) | U.synthetic))]
print('Après filtres :', len(U), f'(-{n0 - len(U)})')

lab = U.task.isin(['sentiment', 'fakenews'])
conflict = U[lab].groupby(['dataset', 'task', 'text']).label.transform('nunique') > 1
conflict_idx = conflict[conflict].index
print('Textes avec labels contradictoires retirés :', len(conflict_idx))
U = U.drop(conflict_idx)

before = len(U)
lab = U.task.isin(['sentiment', 'fakenews'])
U = pd.concat([U[lab].drop_duplicates(['dataset', 'task', 'text']),
               U[U.task.eq('lm')].drop_duplicates('text'),
               U[U.task.eq('parallel')]], ignore_index=True)
print('Doublons retirés :', before - len(U))

flores_texts = set(U.loc[U.dataset.eq('flores_plus'), 'text'])
contam = U.usage.eq('train_ok') & U.text.isin(flores_texts)
print('Textes identiques à FLORES+ retirés :', int(contam.sum()))
U = U[~contam].reset_index(drop=True)
gc.collect()

## 7. Découpage train / val / test (seed 42)
- **sentiment / fake news** : on garde le test d'origine quand il existe (FASSILA, TSAC) et on prend 10 % du train pour la validation ;
  sinon (Algerian Dialect, TUNIZI) découpage **80 / 10 / 10 stratifié par label** ;
- **pré-entraînement** : 98 / 1 / 1 par dataset ;
- **FLORES+** : jamais d'entraînement, `dev` → `val` et `devtest` → `test` ;
- **MADAR** (plus tard) : découpage **par phrase** (`group_id`) pour éviter la fuite entre dialectes.

Enfin, un texte présent dans val ou test est retiré du train de la même tâche.

In [ ]:
def split_80_10_10(df, strat=True):
    y = df.label if strat else None
    tr, rest = train_test_split(df.index, test_size=0.2, random_state=SEED, stratify=y)
    y2 = df.loc[rest, 'label'] if strat else None
    va, te = train_test_split(rest, test_size=0.5, random_state=SEED, stratify=y2)
    return pd.Series('train', index=df.index).mask(df.index.isin(va), 'val').mask(df.index.isin(te), 'test')

U['split'] = None
for (ds, task), g in U[U.task.isin(['sentiment', 'fakenews'])].groupby(['dataset', 'task']):
    if g.orig_split.eq('test').any():
        s = g.orig_split.replace({'dev': 'val'})
        if not s.eq('val').any():
            tr = g.index[s.eq('train')]
            _, va = train_test_split(tr, test_size=0.1, random_state=SEED, stratify=g.loc[tr, 'label'])
            s = s.mask(g.index.isin(va), 'val')
        U.loc[g.index, 'split'] = s
    else:
        U.loc[g.index, 'split'] = split_80_10_10(g)

rng = np.random.default_rng(SEED)
lm = U.task.eq('lm')
r = rng.random(int(lm.sum()))
U.loc[lm, 'split'] = np.where(r < 0.98, 'train', np.where(r < 0.99, 'val', 'test'))
fl = U.dataset.eq('flores_plus')
U.loc[fl, 'split'] = U.loc[fl, 'orig_split'].map({'dev': 'val', 'devtest': 'test'})

# Pas de fuite : un texte de val/test ne doit pas être dans le train de la même tâche
for task in ['sentiment', 'fakenews', 'lm']:
    t = U.task.eq(task)
    held = set(U.loc[t & U.split.isin(['val', 'test']), 'text'])
    leak = t & U.split.eq('train') & U.text.isin(held)
    print(f'{task:10s} fuite train/test retirée : {int(leak.sum())}')
    U = U[~leak]
U = U.reset_index(drop=True)
print(pd.crosstab([U.task, U.dataset], U.split))

## 8. Corpus équilibré pour le tokenizer (Exp. 2)
Le marocain représente ~220 fois plus de mots que le tunisien ou l'algérien. Un tokenizer entraîné sur tout serait « marocain ».
On tire au hasard, **uniquement dans les splits `train`**, le même nombre de mots par dialecte
(= le dialecte le plus petit, sauf si `CAP_WORDS` est fixé). FLORES+ n'y entre jamais.

In [ ]:
CAP_WORDS = None   # ex. 2_000_000 pour plafonner ; None = taille du plus petit dialecte
pool = U[U.split.eq('train') & U.usage.eq('train_ok') & U.dialect.isin(['DZ', 'MA', 'TN'])]
pool = pool.drop_duplicates('text')
avail = pool.groupby('dialect').n_words.sum()
cap = int(min(avail.min(), CAP_WORDS or np.inf))
print('Mots disponibles :', avail.to_dict(), '→ quota par dialecte :', cap)

parts = []
for d, g in pool.groupby('dialect'):
    g = g.sample(frac=1, random_state=SEED)
    parts.append(g[g.n_words.cumsum() <= cap])
tok = pd.concat(parts, ignore_index=True)
tok['text'] = tok.text.str.replace(EMOJI, ' ', regex=True).str.replace(r'\s+', ' ', regex=True).str.strip()
print(tok.groupby(['dialect', 'lang']).n_words.sum().unstack(fill_value=0))

## 9. Sauvegarde

In [ ]:
OUT = ['id', 'text', 'dialect', 'script', 'lang', 'task', 'label', 'label_orig',
       'dataset', 'sub_source', 'split', 'group_id', 'usage', 'n_words']
U['id'] = U.dataset + ':' + U.split.astype(str) + ':' + U.groupby(['dataset', 'split']).cumcount().astype(str)
files = {
    'sentiment.parquet': U[U.task.eq('sentiment')],
    'fakenews.parquet': U[U.task.eq('fakenews')],
    'lm.parquet': U[U.task.eq('lm')],
    'parallel_eval.parquet': U[U.dataset.eq('flores_plus')],
    'tokenizer_balanced.parquet': tok.assign(split='train'),
}
for name, df in files.items():
    df = df.reindex(columns=OUT)
    df.to_parquet(PROC / name, index=False)
    print(f'{name:28s} {len(df):>9,} lignes')

## 10. Récapitulatif (à copier pour Claude)

In [ ]:
recap = (U.groupby(['task', 'dataset', 'dialect', 'split'])
           .agg(lignes=('text', 'size'), mots=('n_words', 'sum'))
           .reset_index())
print(recap.to_markdown(index=False))
print('\nLangue des textes (lang) :')
print(pd.crosstab(U.dataset, U.lang).to_markdown())
print('\nSentiment, labels par split :')
S = U[U.task.eq('sentiment')]
print(pd.crosstab([S.dataset, S.split], S.label).to_markdown())
print('\nCorpus tokenizer (mots) :')
print(tok.groupby('dialect').n_words.sum().to_markdown())

### ✅ Fin de l'étape 3
Copie les quatre tableaux ci-dessus et colle-les à Claude.
Les fichiers de `data/processed/` sont prêts pour les expériences 1 à 4.
Quand MADAR arrivera : relancer `00_donnees.ipynb` puis ce notebook (le découpage MADAR se fera par `group_id`).